# Exploratory data analysis - Lending Club 36-month loans, 2007-2015

EDA only; no modelling here (see `train.py`). Uses the same loaders as the pipeline so the population
shown is exactly the modelling population. Requires the real data (`python scripts/download_data.py`).

In [ ]:
import sys, json
sys.path.insert(0, "../src")
import pandas as pd
import matplotlib.pyplot as plt
from credit_risk.config import Config
from credit_risk.data import load_population, time_split, TARGET
from credit_risk.features import engineer, NUMERIC_FEATURES

cfg = Config()
pop, scope = load_population(cfg)
print(json.dumps(scope, indent=2))
pop.shape

## Default rate and volume by issue year
Volume grows sharply over time; default rates vary by vintage, which is why the split is out-of-time.

In [ ]:
by_year = pop.groupby(pop["issue_d"].dt.year).agg(n=(TARGET, "size"), default_rate=(TARGET, "mean"))
by_year

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(by_year.index, by_year["default_rate"], marker="o", color="#2a78d6", linewidth=2)
ax.set_title("Observed default rate by issue year (36-month loans)", loc="left")
ax.set_ylabel("Default rate"); ax.grid(alpha=0.3)
plt.show()

## Out-of-time split sizes

In [ ]:
train, val, test = time_split(pop, cfg)
pd.DataFrame({name: {"n": len(d), "default_rate": d[TARGET].mean(), "from": d["issue_d"].min().date(),
                     "to": d["issue_d"].max().date()} for name, d in [("train", train), ("val", val), ("test", test)]}).T

## Missingness of application fields (training period)

In [ ]:
eng = engineer(train)
eng[NUMERIC_FEATURES].isna().mean().sort_values(ascending=False).head(12)

## Default rate by FICO band and by LC grade (grade is a benchmark only, never a feature)

In [ ]:
fico_band = pd.cut((train["fico_range_low"] + train["fico_range_high"]) / 2, [600, 660, 680, 700, 720, 740, 780, 850])
display(train.groupby(fico_band, observed=True)[TARGET].agg(["size", "mean"]))
train.groupby("grade")[TARGET].agg(["size", "mean"])

## Default rate by purpose and home ownership

In [ ]:
display(train.groupby("purpose")[TARGET].agg(["size", "mean"]).sort_values("size", ascending=False))
train.groupby("home_ownership")[TARGET].agg(["size", "mean"])